# Mappe da JSON enriched

Notebook separato dalla fase di enrich.

Legge un file JSON già arricchito con `lat` e `lon`, poi crea:
- mappa con confini regionali
- mappa con confini provinciali
- mappa con un layer per ciascuna provincia


In [ ]:
from pathlib import Path
import json
import folium
from IPython.display import display

# Percorsi principali
BASE = Path(".")
INPUT_JSON = BASE / "all_risultati_enriched_2.4.json"   # oppure merged_json.json
GEOJSON_REGIONI = BASE / "geo-json/confini_regioni.geojson"
GEOJSON_PROVINCE = BASE / "geo-json/confini_province.geojson"
OUTPUT_DIR = BASE / "output/mappe"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAPPA_REGIONI = OUTPUT_DIR / "mappa_con_confini_regionali.html"
MAPPA_PROVINCE = OUTPUT_DIR / "mappa_con_confini_provinciali.html"
MAPPA_PROVINCE_LAYER = OUTPUT_DIR / "mappa_province_con_punti_interni.html"

# Centro iniziale
center_lat = 41.9028
center_lon = 12.4964
zoom_start = 6

TILES = "https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png"
ATTR = "&copy; <a href='http://osm.org/copyright'>OpenStreetMap</a>"
REG_NAME = "Regioni"
PRO_NAME = "Province"


In [ ]:
def point_in_ring(lat, lon, ring):
    inside = False
    n = len(ring)
    for i in range(n):
        x1, y1 = ring[i][0], ring[i][1]
        x2, y2 = ring[(i + 1) % n][0], ring[(i + 1) % n][1]
        cond = ((y1 > lat) != (y2 > lat)) and (lon < (x2 - x1) * (lat - y1) / (y2 - y1 + 1e-15) + x1)
        if cond:
            inside = not inside
    return inside

def point_in_polygon(lat, lon, polygon_coords):
    if not polygon_coords:
        return False
    outer = polygon_coords[0]
    if not point_in_ring(lat, lon, outer):
        return False
    for hole in polygon_coords[1:]:
        if point_in_ring(lat, lon, hole):
            return False
    return True

def point_in_multipolygon(lat, lon, multipolygon_coords):
    for poly in multipolygon_coords:
        if point_in_polygon(lat, lon, poly):
            return True
    return False

def point_in_feature(lat, lon, feature):
    geom = feature.get("geometry") or {}
    gtype = geom.get("type")
    coords = geom.get("coordinates", [])
    if gtype == "Polygon":
        return point_in_polygon(lat, lon, coords)
    if gtype == "MultiPolygon":
        return point_in_multipolygon(lat, lon, coords)
    return False

def guess_feature_name(feature, fallback_prefix="feature"):
    props = feature.get("properties") or {}
    candidate_keys = [
        "provincia", "PROVINCIA", "DEN_PROV", "DENOM_PROV", "NOME_PROV", "NAME_2",
        "SIGLA", "SIGLA_PROV",
        "regione", "REGIONE", "DEN_REG", "DENOM_REG", "NOME_REG", "NAME_1",
        "name", "Name", "NOME", "DENOMINAZIONE", "DEN_UTS", "DENOM"
    ]
    for k in candidate_keys:
        if k in props and props[k]:
            return str(props[k])
    fid = feature.get("id")
    if fid is not None:
        return f"{fallback_prefix}_{fid}"
    for k, v in props.items():
        if v:
            return f"{fallback_prefix}_{str(v)[:24]}"
    return f"{fallback_prefix}_sconosciuto"


In [ ]:
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    merged = json.load(f)

with open(GEOJSON_REGIONI, "r", encoding="utf-8") as f:
    geo_regioni = json.load(f)

with open(GEOJSON_PROVINCE, "r", encoding="utf-8") as f:
    geo_province = json.load(f)

punti = []
for item in merged:
    for ent in item.get("entities", []):
        lat = ent.get("lat")
        lon = ent.get("lon")
        if isinstance(lat, (int, float)) and isinstance(lon, (int, float)):
            punti.append({
                "lat": float(lat),
                "lon": float(lon),
                "nome": ent.get("nome", ""),
                "comune": ent.get("comune", ""),
                "provincia": ent.get("provincia", ""),
                "regione": ent.get("regione", ""),
            })

print(f"Punti caricati: {len(punti)}")
if not punti:
    raise SystemExit("Nessun punto con lat/lon trovato nel file JSON enriched.")


## Mappa 1 — confini regionali + punti

In [ ]:
m_regioni = folium.Map(
    location=(center_lat, center_lon),
    zoom_start=zoom_start,
    tiles=None,
    control_scale=True,
)

folium.TileLayer(tiles=TILES, attr=ATTR, name=REG_NAME).add_to(m_regioni)

folium.GeoJson(
    geo_regioni,
    name="Confini regionali",
    style_function=lambda feat: {"fillColor": "transparent", "color": "#333333", "weight": 1},
).add_to(m_regioni)

fg_punti_reg = folium.FeatureGroup(name=f"Comuni (tutti i punti) [{len(punti)}]", show=True)
for p in punti:
    lat, lon = p.get("lat"), p.get("lon")
    if isinstance(lat, (int, float)) and isinstance(lon, (int, float)):
        folium.CircleMarker(
            location=(lat, lon),
            radius=5,
            color="#d62728",
            fill=True,
            fill_opacity=0.6,
            popup=folium.Popup(
                f"Comune: {p['comune']}<br/>Provincia: {p['provincia']}<br/>Regione: {p['regione']}",
                max_width=300,
            ),
        ).add_to(fg_punti_reg)

fg_punti_reg.add_to(m_regioni)
folium.LayerControl(collapsed=False).add_to(m_regioni)

m_regioni.save(MAPPA_REGIONI)
m_regioni


## Mappa 2 — confini provinciali + punti

In [ ]:
m_province = folium.Map(
    location=(center_lat, center_lon),
    zoom_start=zoom_start,
    control_scale=True,
    tiles=None,
)

folium.TileLayer(tiles=TILES, attr=ATTR, name=PRO_NAME).add_to(m_province)

folium.GeoJson(
    geo_province,
    name="Confini provinciali",
    style_function=lambda feat: {"fillColor": "transparent", "color": "#666666", "weight": 1},
).add_to(m_province)

fg_punti_prov = folium.FeatureGroup(name="Comuni (tutti i punti)", show=True)
for p in punti:
    folium.CircleMarker(
        location=(p["lat"], p["lon"]),
        radius=5,
        color="#1f77b4",
        fill=True,
        fill_opacity=0.6,
        popup=folium.Popup(
            f"<b>{p['nome']}</b><br/>Comune: {p['comune']}<br/>Provincia: {p['provincia']}<br/>Regione: {p['regione']}",
            max_width=300,
        ),
    ).add_to(fg_punti_prov)

fg_punti_prov.add_to(m_province)
folium.LayerControl(collapsed=False).add_to(m_province)

m_province.save(MAPPA_PROVINCE)
m_province


## Mappa 3 — layer separato per provincia

In [ ]:
m_province_layers = folium.Map(
    location=(center_lat, center_lon),
    zoom_start=zoom_start,
    control_scale=True,
    tiles=None,
)

folium.TileLayer(tiles=TILES, attr=ATTR, name=PRO_NAME).add_to(m_province_layers)

folium.GeoJson(
    geo_province,
    name="Confini provinciali (sfondo)",
    style_function=lambda feat: {"fillColor": "transparent", "color": "#aaaaaa", "weight": 0.8},
).add_to(m_province_layers)

features_prov = geo_province.get("features", [])
for feat in features_prov:
    prov_name = guess_feature_name(feat, fallback_prefix="prov")
    fg = folium.FeatureGroup(name=f"Provincia: {prov_name}", show=False)

    folium.GeoJson(
        feat,
        name=f"Confine {prov_name}",
        style_function=lambda f: {"fillColor": "transparent", "color": "#000000", "weight": 2},
        highlight_function=lambda f: {"weight": 3, "color": "#111111"},
        tooltip=prov_name,
    ).add_to(fg)

    for p in punti:
        if point_in_feature(p["lat"], p["lon"], feat):
            folium.CircleMarker(
                location=(p["lat"], p["lon"]),
                radius=5,
                color="#2ca02c",
                fill=True,
                fill_opacity=0.7,
                popup=folium.Popup(
                    f"<b>{p['nome']}</b><br/>Comune: {p['comune']}<br/>Provincia: {p['provincia']}<br/>Regione: {p['regione']}",
                    max_width=300,
                ),
            ).add_to(fg)

    fg.add_to(m_province_layers)

folium.LayerControl(collapsed=False).add_to(m_province_layers)

m_province_layers.save(MAPPA_PROVINCE_LAYER)
m_province_layers


## File generati

In [ ]:
print("Creati:")
print(MAPPA_REGIONI)
print(MAPPA_PROVINCE)
print(MAPPA_PROVINCE_LAYER)
